<a id="top"></a>

# Lab L0208: few-shot examples and their cost

```yaml
title: "Lab L0208: few-shot examples and their cost"
keywords: few-shot, in-context examples, alternating turns, single block, token cost, lab
estimated duration: 30
```

> **Lesson:** L02. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L02/objectives.md). Solutions: `L0208_lab_solutions.ipynb`. Pure Python — no API key needed.
>
> **After this lab you can:** build a few-shot prompt two ways; estimate its per-call token cost; recognize when few-shot is the wrong tool.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — Few-shot as alternating turns](#2-problem-1--few-shot-as-alternating-turns)
- [3. Problem 2 — Few-shot as a single block](#3-problem-2--few-shot-as-a-single-block)
- [4. Problem 3 — The per-call cost of examples](#4-problem-3--the-per-call-cost-of-examples)
- [5. Problem 4 — When is few-shot the wrong tool?](#5-problem-4--when-is-few-shot-the-wrong-tool)
- [6. Self-check](#6-self-check)

## 1. Setup

In [1]:
from fluffy_potato_curriculum.potato_llm import Message

# A few worked examples for a sentiment task, as (input, label) pairs.
EXAMPLES: list[tuple[str, str]] = [
    ("The build is broken again.", "negative"),
    ("Thanks, this fixed it perfectly!", "positive"),
    ("It works but the docs are unclear.", "mixed"),
]
REAL_INPUT = "The new release is fast, though setup took a while."


def est_tokens(text: str) -> int:
    """Rough token estimate: ~4 characters per token."""
    return len(text) // 4


print("setup ready")

setup ready


[↑ Back to top](#top)

## 2. Problem 1 — Few-shot as alternating turns

Implement `build_fewshot_turns`: turn each `(input, label)` example into a `user` turn (the input) followed by an `assistant` turn (the label), then append the real input as a final `user` turn. Return the `messages` list.

Note the surprise: the **assistant** role is the right home for the *fake* example outputs.

In [2]:
def build_fewshot_turns(examples: list[tuple[str, str]], real_input: str) -> list[Message]:
    messages: list[Message] = []
    for text, label in examples:
        messages.append(Message.user(text))
        messages.append(Message.assistant(label))
    messages.append(Message.user(real_input))
    return messages


turns = build_fewshot_turns(EXAMPLES, REAL_INPUT)
for m in turns:
    print(f"{m.role:9} | {m.content}")
print("\nlength:", len(turns))  # expect 2*len(EXAMPLES) + 1

user      | The build is broken again.
assistant | negative
user      | Thanks, this fixed it perfectly!
assistant | positive
user      | It works but the docs are unclear.
assistant | mixed
user      | The new release is fast, though setup took a while.

length: 7


[↑ Back to top](#top)

## 3. Problem 2 — Few-shot as a single block

Implement `build_fewshot_block`: format all examples as one labeled text block (e.g. `"input -> label"` lines), then append the real input. Return a `messages` list with a **single** `user` message. Same examples, different placement.

In [3]:
def build_fewshot_block(examples: list[tuple[str, str]], real_input: str) -> list[Message]:
    lines = [f"{text} -> {label}" for text, label in examples]
    body = "Examples:\n" + "\n".join(lines) + f"\n\nNow classify: {real_input}"
    return [Message.user(body)]


block = build_fewshot_block(EXAMPLES, REAL_INPUT)
print("messages:", len(block))  # expect 1
print(block[0].content)

messages: 1
Examples:
The build is broken again. -> negative
Thanks, this fixed it perfectly! -> positive
It works but the docs are unclear. -> mixed

Now classify: The new release is fast, though setup took a while.


[↑ Back to top](#top)

## 4. Problem 3 — The per-call cost of examples

Few-shot is paid for on every call. Implement `fewshot_input_tokens`: estimate the input tokens added by the examples alone (sum `est_tokens` over each example's input and label). Then show how the cost scales as you add examples.

In [4]:
def fewshot_input_tokens(examples: list[tuple[str, str]]) -> int:
    return sum(est_tokens(text) + est_tokens(label) for text, label in examples)


for n in range(1, len(EXAMPLES) + 1):
    print(f"{n} example(s): ~{fewshot_input_tokens(EXAMPLES[:n])} extra input tok / call")

1 example(s): ~8 extra input tok / call
2 example(s): ~18 extra input tok / call
3 example(s): ~27 extra input tok / call


Every example is re-sent on **every** call — the cost is linear in the number of examples and paid forever. That is the dial you weigh against the quality gain.

[↑ Back to top](#top)

## 5. Problem 4 — When is few-shot the wrong tool?

For each situation, write `few-shot` or `not few-shot` and one phrase of justification.

| Situation | few-shot? | Why |
| --- | --- | --- |
| A single clear instruction already classifies correctly | not few-shot | a clear instruction already works — skip the per-call token cost |
| The team uses idiosyncratic label wording the model can't guess | few-shot | show the exact wording; the model can't guess a team convention |
| The task needs multi-step reasoning, not pattern-matching | not few-shot | reasoning wants chain-of-thought (L06), not more examples |
| The example set would fill most of the context window | not few-shot | examples would dominate the window — reach for retrieval (L21) or another approach |

## 6. Self-check

Compare against `L0208_lab_solutions.ipynb`. Done when the alternating-turns list has length `2*len(EXAMPLES)+1`, the block version has a single user message, and your cost grows with the example count.

[↑ Back to top](#top)